# HornBro (FSE 2025) on the repair benchmark, same 60 circuits

HornBro needs Linux and Python 3.10 with its own dependencies, which conflict with this repository's Qiskit, so the comparison runs in two environments that exchange OpenQASM files:
1. **HornBro environment**: HornBro repairs each circuit given the reference and the faulty circuit (its own design).
2. **This repository's environment**: the shared acceptance predicate judges what HornBro returned.

HornBro is MIT-licensed, https://github.com/JanusQ/HornBro. Its README asks for 8+ cores and 32 GB RAM; a free Colab session has less, so this run uses 60 small circuits and one process at a time. If something fails, paste the error text back; the adapter records crashes per circuit instead of stopping.

Runtime -> Change runtime type -> **CPU is enough**.

In [ ]:
import os, subprocess
def sh(cmd, check=True):
    print('$', cmd)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    print((r.stdout + r.stderr)[-2500:])
    if check and r.returncode: raise RuntimeError('command failed')
os.chdir('/content')
if not os.path.exists('physics-constrained-quantum-repair'):
    sh('git clone --depth 1 https://github.com/alok-108/physics-constrained-quantum-repair.git')
if not os.path.exists('HornBro'):
    sh('git clone https://github.com/JanusQ/HornBro.git')
sh('cd HornBro && git log --oneline | head -1')

## 1. Python 3.10 environment for HornBro (about 5-10 minutes)

In [ ]:
sh('pip -q install uv')
sh('uv venv --python 3.10 /content/hb310')
sh('cd /content/HornBro && uv pip install -q -p /content/hb310/bin/python -e . qiskit_qasm3_import "qiskit>=1.3,<2" "qiskit-aer>=0.15,<0.17"')
sh('/content/hb310/bin/python -c "import qiskit, hornbro; print(qiskit.__version__)"')
sh('cd /content/HornBro && PYTHONPATH=/content/HornBro /content/hb310/bin/python -c "import evaluation.efficiencyHornBro as H; print(\'HornBro imports fine\')"')

## 2. Run HornBro on the 60 circuits (resumable)

In [ ]:
env = dict(os.environ, PYTHONPATH='/content/HornBro', RAY_DEDUP_LOGS='0', OMP_NUM_THREADS='1')
cmd = ('cd /content/physics-constrained-quantum-repair && /content/hb310/bin/python src/hornbro_adapter.py '
       '--hornbro /content/HornBro --limit 60 --out /content/hornbro_out')
p = subprocess.Popen(cmd, shell=True, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout: print(line.rstrip())
p.wait()

## 3. Judge the returned circuits with the shared predicate

In [ ]:
sh('pip -q install -r /content/physics-constrained-quantum-repair/requirements.txt')
sh('cd /content/physics-constrained-quantum-repair && python src/score_hornbro.py --dir /content/hornbro_out --out /content/results_hornbro.csv')

## 4. Download
Download `results_hornbro.csv` and the folder `hornbro_out` (zip) and send them back; they go into `experiments/` and the comparison tables.

In [ ]:
sh('cd /content && zip -qr hornbro_out.zip hornbro_out results_hornbro.csv')
try:
    from google.colab import files
    files.download('/content/hornbro_out.zip')
except Exception as e:
    print('zip is at /content/hornbro_out.zip', e)